In [1]:
# -*- coding: utf-8 -*-
import os
import time
import json
import ee
import geemap
import pandas as pd

In [2]:
# ============== 用户配置 ==============
CSV_PATH = r"E:\NWP\CS2_S1_matched\time_match_2023.csv"   # 你的CSV
SCENENAME_COL = "sceneName"                                # CSV中场景名列名；若你列叫 'sceneName' 就不用改
DRIVE_FOLDER = "S1_EW_exports"                             # Google Drive 目标文件夹
SCALE_M = 40                                               # EW 建议 40 m
MAX_EXPORTS = 500                                          # 最多发起的导出任务数（防止一次性太多

In [2]:
# 初始化 Earth Engine
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()


Successfully saved authorization token.


In [4]:
# 读取CSV，抽取独特的 sceneName
df = pd.read_csv(CSV_PATH)
if SCENENAME_COL not in df.columns:
    raise ValueError(f"CSV中未找到列：{SCENENAME_COL}")
scene_names = (
    df[SCENENAME_COL]
    .astype(str)
    .str.strip()
    .dropna()
    .unique()
    .tolist()
)

print(f"[INFO] 从CSV读取到 {len(scene_names)} 条 sceneName")

[INFO] 从CSV读取到 344 条 sceneName


In [5]:
def get_s1_image_by_scene(scene_name: str) -> ee.Image:
    """按 system:index 精确筛选 S1 GRD EW 影像；返回 ee.Image 或 None"""
    col = (ee.ImageCollection('COPERNICUS/S1_GRD')
           .filter(ee.Filter.eq('system:index', scene_name))
           .filter(ee.Filter.eq('productType', 'GRD'))
           .filter(ee.Filter.eq('instrumentMode', 'EW')))
    img = col.first()
    return ee.Image(img) if img else None

def pick_hh_hv_bands(img: ee.Image):
    """返回该影像中可用的 HH/HV 波段列表（可能只有HH或HV，也可能两者都有）"""
    band_names = ee.List(img.bandNames())
    bands = []
    # Sentinel-1在GEE集合里波段名就是 'HH' / 'HV' / 'VV' / 'VH'
    if band_names.contains('HH').getInfo():
        bands.append('HH')
    if band_names.contains('HV').getInfo():
        bands.append('HV')
    return bands

def export_image_to_drive(img: ee.Image, scene_name: str, bands: list):
    """将所选波段导出到Google Drive；返回导出任务对象与任务描述"""
    desc = f"{scene_name}_EW_" + "_".join(bands)
    task = ee.batch.Export.image.toDrive(
        image = img.select(bands),
        description = desc,
        folder = DRIVE_FOLDER,
        fileNamePrefix = desc,
        region = img.geometry(),  # 使用影像自身范围；需要AOI可替换为你的多边形
        scale = SCALE_M,
        maxPixels = 1e13
    )
    task.start()
    return task, desc

In [6]:
# 汇总信息表（本地保存）
summary_rows = []

launched = 0
for i, scene in enumerate(scene_names, 1):
    print(f"\n[{i}/{len(scene_names)}] {scene}")
    try:
        img = get_s1_image_by_scene(scene)
        if img is None:
            print("  [MISS] GEE 找不到该 sceneName")
            summary_rows.append({
                "sceneName": scene,
                "status": "NOT_FOUND",
                "export_description": "",
                "bands_exported": "",
                "startTime": "",
                "orbitPass": "",
                "relativeOrbit": "",
                "polarisation": "",
                "notes": "system:index 未匹配"
            })
            continue

        # 可用极化（只取HH/HV）
        bands = pick_hh_hv_bands(img)
        if not bands:
            print("  [MISS] 该影像不含 HH/HV 波段（可能是VV/VH）")
            summary_rows.append({
                "sceneName": scene,
                "status": "NO_HH_HV",
                "export_description": "",
                "bands_exported": "",
                "startTime": img.get('system:time_start').getInfo(),
                "orbitPass": img.get('orbitProperties_pass').getInfo(),
                "relativeOrbit": img.get('relativeOrbitNumber_start').getInfo(),
                "polarisation": ",".join(img.get('transmitterReceiverPolarisation').getInfo()),
                "notes": "仅含VV/VH或其它"
            })
            continue

        # 导出
        if launched >= MAX_EXPORTS:
            print("  [STOP] 已达到 MAX_EXPORTS 上限，停止创建新任务。")
            break

        task, desc = export_image_to_drive(img, scene, bands)
        launched += 1

        # 拉取关键信息做登记（getInfo 调用较慢，但便于留档）
        props = {
            "startTime": img.get('system:time_start').getInfo(),
            "orbitPass": img.get('orbitProperties_pass').getInfo(),
            "relativeOrbit": img.get('relativeOrbitNumber_start').getInfo(),
            "polarisation": ",".join(img.get('transmitterReceiverPolarisation').getInfo()),
            "sliceNumber": img.get('sliceNumber').getInfo() if img.get('sliceNumber') else ""
        }

        summary_rows.append({
            "sceneName": scene,
            "status": "EXPORTED",
            "export_description": desc,
            "bands_exported": ",".join(bands),
            **props
        })

        print(f"  [OK] 已提交导出任务：{desc}  -> Drive/{DRIVE_FOLDER}")
        time.sleep(0.3)  # 小憩，避免速率过高

    except Exception as e:
        print(f"  [ERR] {e}")
        summary_rows.append({
            "sceneName": scene,
            "status": "ERROR",
            "export_description": "",
            "bands_exported": "",
            "startTime": "",
            "orbitPass": "",
            "relativeOrbit": "",
            "polarisation": "",
            "notes": str(e)
        })

# 保存本地汇总 CSV
summary_df = pd.DataFrame(summary_rows)
out_csv = os.path.join(os.path.dirname(CSV_PATH), "gee_s1_ew_export_summary.csv")
summary_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n[DONE] 已保存导出登记表：{out_csv}")
print(f"[INFO] 发起导出任务数：{launched}（请在 GEE Tasks 面板查看进度）")


[1/344] S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378
  [OK] 已提交导出任务：S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378_EW_HH_HV  -> Drive/S1_EW_exports

[2/344] S1A_EW_GRDM_1SDH_20230105T212255_20230105T212334_046654_05978A_50D1
  [OK] 已提交导出任务：S1A_EW_GRDM_1SDH_20230105T212255_20230105T212334_046654_05978A_50D1_EW_HH_HV  -> Drive/S1_EW_exports

[3/344] S1A_IW_GRDH_1SDH_20230105T194528_20230105T194553_046653_059785_C4E7
  [ERR] Image.bandNames: Parameter 'image' is required and may not be null.

[4/344] S1A_EW_GRDM_1SDH_20230630T134853_20230630T134953_049216_05EB02_4620
  [OK] 已提交导出任务：S1A_EW_GRDM_1SDH_20230630T134853_20230630T134953_049216_05EB02_4620_EW_HH_HV  -> Drive/S1_EW_exports

[5/344] S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
  [OK] 已提交导出任务：S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83_EW_HH_HV  -> Drive/S1_EW_exports

[6/344] S1A_EW_GRDM_1SDH_20230630T120922_20230630T121027_049215_05EAFB_195